# 第二篇论文：有限效率 PNRD 下的成功概率与保真度

本 notebook 将第一篇论文 Table V 的两级有限效率 PNRD 模型，适配到第二篇论文的相干态/猫态 OPA synthesizer。

它同时输出两种概率：

$$
P_{\rm trial}^{\rm simple}=\eta_{\rm out}P_{\rm Fock}(m,g')P_{\psi}(g,m,n),
$$

用于复现第二篇 Tables I-IV 的原有口径，以及

$$
P_{\rm trial}^{(\eta_{\rm prep},\eta_{\rm out})}(m,n)
=\sum_{j=m}^{\infty}\sum_{l=n}^{\infty}
w_{\eta_{\rm prep}}(j\to m)P_{\rm Fock}(j,g')
w_{\eta_{\rm out}}(l\to n)P_{\psi}(g,j,l),
$$

即两个 PNRD 都具有有限效率时的真实注册概率。

同时，本 notebook 构造登记到 $(m,n)$ 后的条件混态，并计算它对固定目标猫态的保真度。它因此可以直接用于讨论探测效率从 $0.90$ 改为 $0.95$ 时，对概率和保真度的影响。

## 模型卡

- **目的**：在第二篇论文的猫态输入构型中，计算简化成功概率、两级有限效率成功概率和有限效率条件保真度。
- **子系统与基底**：只显式表示信号模式；采用截断 Fock 基，维数 `N_DIM`。idler 模式由解析条件操作和 PNRD 计数指标 $(j,l)$ 表示。
- **输入态**：归一化 OCS 或 ECS；辅助 SPDC 以 $P_{\rm Fock}(j,g')$ 提供实际 idler Fock 数 $j$。
- **条件操作**：第二篇论文式 (3)-(4)；主 OPA 后以有限效率 PNRD 登记 $n$ 光子。
- **输出量**：$P_{\rm trial}^{\rm simple}$、$P_{\rm trial}^{(\eta_{\rm prep},\eta_{\rm out})}$、有限效率条件保真度和错误通道贡献。
- **近似与限制**：独立光子损耗、无暗计数、无误分辨串扰；有限和以 `J_MAX`、`L_MAX` 截断，必须做截断收敛检查。
- **确定性**：无随机采样。
- **独立检查**：令 `ETA_PREP=ETA_OUT=1.0` 时，完整模型应回到 $P_{\rm Fock}(m,g')P_{\psi}(g,m,n)$。

In [1]:
# 1. 导入依赖
from __future__ import annotations

import csv
import math
from pathlib import Path

import numpy as np
import qutip as qt


In [2]:
# 2. EDIT HERE: 物理参数、探测效率和数值截断
# 第二篇论文原概率代码的约定：g_prep=1.0483，对应 tanh(rho_prime)≈0.30。
G_PREP = 1.0483

# 两个 PNRD 的效率。当前均设为原代码的 0.90。
# 若要研究 95% 效率，将这两行都改成 0.95，然后从头运行 notebook。
ETA_PREP = 0.90   # 辅助 SPDC / Fock 态制备 heralding PNRD
ETA_OUT = 0.90    # 主 OPA 输出端 PNRD

# 截断参数。修改后必须比较更大截断的结果，确认 P_trial 和 F 已收敛。
N_DIM = 50
J_MAX = 8         # 辅助 SPDC 实际产生的最大光子数 j
L_MAX = 14        # 主 OPA 输出端实际光子数 l
TOP_CHANNELS_PER_CASE = 0  # 设为 5 可在屏幕打印每个构型最重要的 5 个真实 (j,l) 通道；完整数据始终保存至 CSV。

# 输出文件目录。每次运行覆盖同名 CSV，便于检查当前参数的结果。
OUTPUT_DIR = Path("outputs") / "second_paper_finite_efficiency_probability_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 这些行逐行转录自第二篇论文 Tables I-IV。
# target_* 只用于有限效率后的条件保真度；不影响成功概率。
# Table I/IV 的 m=0 行沿用论文表格的辅助 SPDC 真空概率 P_Fock(0,g_prime)。
CASES = [
    # Table II: odd cat state (OCS), catalytic configurations
    {"label": "Table II OCS (1,1), alpha=0.30", "m": 1, "n": 1, "g": 1.60, "alpha_in": 0.30, "input_parity": "odd", "target_alpha": 0.58, "target_gamma": 0.030, "target_parity": "odd", "paper_p_trial": 1.39e-3, "paper_fidelity": 1.000},
    {"label": "Table II OCS (1,1), alpha=1.06", "m": 1, "n": 1, "g": 1.60, "alpha_in": 1.06, "input_parity": "odd", "target_alpha": 2.02, "target_gamma": 0.340, "target_parity": "odd", "paper_p_trial": 2.73e-3, "paper_fidelity": 0.992},
    {"label": "Table II OCS (1,1), alpha=1.31", "m": 1, "n": 1, "g": 1.60, "alpha_in": 1.31, "input_parity": "odd", "target_alpha": 2.31, "target_gamma": 0.390, "target_parity": "odd", "paper_p_trial": 3.94e-3, "paper_fidelity": 0.976},
    {"label": "Table II OCS (2,2), alpha=0.30", "m": 2, "n": 2, "g": 1.36, "alpha_in": 0.30, "input_parity": "odd", "target_alpha": 0.46, "target_gamma": 0.030, "target_parity": "odd", "paper_p_trial": 2.78e-4, "paper_fidelity": 1.000},
    {"label": "Table II OCS (2,2), alpha=1.06", "m": 2, "n": 2, "g": 1.36, "alpha_in": 1.06, "input_parity": "odd", "target_alpha": 1.67, "target_gamma": 0.310, "target_parity": "odd", "paper_p_trial": 3.33e-4, "paper_fidelity": 0.999},
    {"label": "Table II OCS (2,2), alpha=1.31", "m": 2, "n": 2, "g": 1.36, "alpha_in": 1.31, "input_parity": "odd", "target_alpha": 2.11, "target_gamma": 0.450, "target_parity": "odd", "paper_p_trial": 3.71e-4, "paper_fidelity": 0.995},
    {"label": "Table II OCS (3,3), alpha=0.30", "m": 3, "n": 3, "g": 4.50, "alpha_in": 0.30, "input_parity": "odd", "target_alpha": 0.68, "target_gamma": 0.050, "target_parity": "odd", "paper_p_trial": 5.73e-7, "paper_fidelity": 1.000},
    {"label": "Table II OCS (3,3), alpha=1.06", "m": 3, "n": 3, "g": 4.50, "alpha_in": 1.06, "input_parity": "odd", "target_alpha": 2.30, "target_gamma": 0.430, "target_parity": "odd", "paper_p_trial": 1.66e-6, "paper_fidelity": 0.980},
    {"label": "Table II OCS (3,3), alpha=1.31", "m": 3, "n": 3, "g": 4.50, "alpha_in": 1.31, "input_parity": "odd", "target_alpha": 2.56, "target_gamma": 0.470, "target_parity": "odd", "paper_p_trial": 2.64e-6, "paper_fidelity": 0.958},
    {"label": "Table II OCS (4,4), alpha=0.30", "m": 4, "n": 4, "g": 2.60, "alpha_in": 0.30, "input_parity": "odd", "target_alpha": 0.47, "target_gamma": 0.030, "target_parity": "odd", "paper_p_trial": 2.74e-7, "paper_fidelity": 1.000},
    {"label": "Table II OCS (4,4), alpha=1.06", "m": 4, "n": 4, "g": 2.60, "alpha_in": 1.06, "input_parity": "odd", "target_alpha": 1.72, "target_gamma": 0.320, "target_parity": "odd", "paper_p_trial": 3.41e-7, "paper_fidelity": 0.999},
    {"label": "Table II OCS (4,4), alpha=1.31", "m": 4, "n": 4, "g": 2.60, "alpha_in": 1.31, "input_parity": "odd", "target_alpha": 2.24, "target_gamma": 0.480, "target_parity": "odd", "paper_p_trial": 3.94e-7, "paper_fidelity": 0.998},

    # Table III: even cat state (ECS), catalytic configurations
    {"label": "Table III ECS (1,1), alpha=0.50", "m": 1, "n": 1, "g": 1.10, "alpha_in": 0.50, "input_parity": "even", "target_alpha": 0.37, "target_gamma": 0.015, "target_parity": "even", "paper_p_trial": 4.91e-2, "paper_fidelity": 1.000},
    {"label": "Table III ECS (1,1), alpha=1.00", "m": 1, "n": 1, "g": 1.10, "alpha_in": 1.00, "input_parity": "even", "target_alpha": 0.74, "target_gamma": 0.061, "target_parity": "even", "paper_p_trial": 3.64e-2, "paper_fidelity": 1.000},
    {"label": "Table III ECS (1,1), alpha=1.50", "m": 1, "n": 1, "g": 1.10, "alpha_in": 1.50, "input_parity": "even", "target_alpha": 1.13, "target_gamma": 0.149, "target_parity": "even", "paper_p_trial": 1.67e-2, "paper_fidelity": 0.997},
    {"label": "Table III ECS (2,2), alpha=0.50", "m": 2, "n": 2, "g": 2.80, "alpha_in": 0.50, "input_parity": "even", "target_alpha": 0.85, "target_gamma": 0.070, "target_parity": "even", "paper_p_trial": 1.62e-5, "paper_fidelity": 1.000},
    {"label": "Table III ECS (2,2), alpha=1.00", "m": 2, "n": 2, "g": 2.80, "alpha_in": 1.00, "input_parity": "even", "target_alpha": 1.71, "target_gamma": 0.246, "target_parity": "even", "paper_p_trial": 4.43e-5, "paper_fidelity": 0.999},
    {"label": "Table III ECS (2,2), alpha=1.50", "m": 2, "n": 2, "g": 2.80, "alpha_in": 1.50, "input_parity": "even", "target_alpha": 2.43, "target_gamma": 0.397, "target_parity": "even", "paper_p_trial": 9.49e-5, "paper_fidelity": 0.991},
    {"label": "Table III ECS (3,3), alpha=0.50", "m": 3, "n": 3, "g": 1.93, "alpha_in": 0.50, "input_parity": "even", "target_alpha": 0.73, "target_gamma": 0.016, "target_parity": "even", "paper_p_trial": 3.33e-6, "paper_fidelity": 1.000},
    {"label": "Table III ECS (3,3), alpha=1.00", "m": 3, "n": 3, "g": 1.93, "alpha_in": 1.00, "input_parity": "even", "target_alpha": 1.51, "target_gamma": 0.262, "target_parity": "even", "paper_p_trial": 5.70e-6, "paper_fidelity": 0.999},
    {"label": "Table III ECS (3,3), alpha=1.50", "m": 3, "n": 3, "g": 1.93, "alpha_in": 1.50, "input_parity": "even", "target_alpha": 2.70, "target_gamma": 0.661, "target_parity": "even", "paper_p_trial": 8.60e-6, "paper_fidelity": 0.981},
    {"label": "Table III ECS (4,4), alpha=0.50", "m": 4, "n": 4, "g": 1.65, "alpha_in": 0.50, "input_parity": "even", "target_alpha": 0.74, "target_gamma": 0.076, "target_parity": "even", "paper_p_trial": 3.85e-7, "paper_fidelity": 1.000},
    {"label": "Table III ECS (4,4), alpha=1.00", "m": 4, "n": 4, "g": 1.65, "alpha_in": 1.00, "input_parity": "even", "target_alpha": 1.53, "target_gamma": 0.309, "target_parity": "even", "paper_p_trial": 6.41e-7, "paper_fidelity": 0.999},
    {"label": "Table III ECS (4,4), alpha=1.50", "m": 4, "n": 4, "g": 1.65, "alpha_in": 1.50, "input_parity": "even", "target_alpha": 2.63, "target_gamma": 0.688, "target_parity": "even", "paper_p_trial": 8.84e-7, "paper_fidelity": 0.984},

    # Table IV: non-catalytic configurations
    {"label": "Table IV ECS (1,2), alpha=1.18", "m": 1, "n": 2, "g": 1.05, "alpha_in": 1.18, "input_parity": "even", "target_alpha": 2.02, "target_gamma": 0.23, "target_parity": "odd", "paper_p_trial": 1.74e-2, "paper_fidelity": 0.996},
    {"label": "Table IV ECS (1,3), alpha=1.97", "m": 1, "n": 3, "g": 1.01, "alpha_in": 1.97, "input_parity": "even", "target_alpha": 3.45, "target_gamma": 0.26, "target_parity": "even", "paper_p_trial": 1.13e-3, "paper_fidelity": 0.991},
    {"label": "Table IV ECS (0,2), alpha=1.97", "m": 0, "n": 2, "g": 1.01, "alpha_in": 1.97, "input_parity": "even", "target_alpha": 3.46, "target_gamma": 0.26, "target_parity": "even", "paper_p_trial": 4.59e-3, "paper_fidelity": 0.991},
    {"label": "Table IV ECS (3,2), alpha=1.45", "m": 3, "n": 2, "g": 1.60, "alpha_in": 1.45, "input_parity": "even", "target_alpha": 2.00, "target_gamma": 0.49, "target_parity": "odd", "paper_p_trial": 1.34e-5, "paper_fidelity": 0.996},
    {"label": "Table IV OCS (1,2), alpha=1.97", "m": 1, "n": 2, "g": 1.05, "alpha_in": 1.97, "input_parity": "odd", "target_alpha": 2.69, "target_gamma": 0.20, "target_parity": "even", "paper_p_trial": 2.20e-2, "paper_fidelity": 0.991},
    {"label": "Table IV OCS (0,2), alpha=1.97", "m": 0, "n": 2, "g": 1.01, "alpha_in": 1.97, "input_parity": "odd", "target_alpha": 3.46, "target_gamma": 0.25, "target_parity": "odd", "paper_p_trial": 4.59e-3, "paper_fidelity": 0.991},
    {"label": "Table IV OCS (3,2), alpha=1.76", "m": 3, "n": 2, "g": 3.30, "alpha_in": 1.76, "input_parity": "odd", "target_alpha": 2.31, "target_gamma": 0.35, "target_parity": "even", "paper_p_trial": 5.52e-6, "paper_fidelity": 0.993},
    {"label": "Table IV OCS (4,2), alpha=1.34", "m": 4, "n": 2, "g": 1.70, "alpha_in": 1.34, "input_parity": "odd", "target_alpha": 2.15, "target_gamma": 0.51, "target_parity": "odd", "paper_p_trial": 2.59e-7, "paper_fidelity": 0.993},

    # Table I: cubic phase states (CPS), coherent-state input
    # alpha_in and correction_displacement are purely imaginary, as in Table I.
    {"label": "Table I CPS (0,4)", "family": "cps", "m": 0, "n": 4, "g": 1.46, "alpha_in": -2.145j, "target_cubicity": 0.103, "target_squeezing": 0.00, "correction_squeezing": 0.521, "correction_displacement": 2.963j, "paper_p_trial": 8.28e-2, "paper_fidelity": 0.997},
    {"label": "Table I CPS (1,1)", "family": "cps", "m": 1, "n": 1, "g": 1.08, "alpha_in": -4.136j, "target_cubicity": 0.100, "target_squeezing": 0.00, "correction_squeezing": 0.203, "correction_displacement": 4.406j, "paper_p_trial": 1.17e-2, "paper_fidelity": 0.975},
    {"label": "Table I CPS (1,4)", "family": "cps", "m": 1, "n": 4, "g": 1.15, "alpha_in": -1.448j, "target_cubicity": 0.146, "target_squeezing": -0.01, "correction_squeezing": 0.589, "correction_displacement": 2.493j, "paper_p_trial": 8.82e-3, "paper_fidelity": 0.989},
    {"label": "Table I CPS (1,5)", "family": "cps", "m": 1, "n": 5, "g": 1.07, "alpha_in": -2.260j, "target_cubicity": 0.100, "target_squeezing": 0.00, "correction_squeezing": 0.469, "correction_displacement": 3.401j, "paper_p_trial": 3.75e-4, "paper_fidelity": 0.991},
    {"label": "Table I CPS (2,2)", "family": "cps", "m": 2, "n": 2, "g": 2.06, "alpha_in": -3.676j, "target_cubicity": 0.100, "target_squeezing": 0.00, "correction_squeezing": 0.399, "correction_displacement": 2.786j, "paper_p_trial": 1.13e-5, "paper_fidelity": 0.993},
    {"label": "Table I CPS (2,5)", "family": "cps", "m": 2, "n": 5, "g": 1.05, "alpha_in": -1.614j, "target_cubicity": 0.100, "target_squeezing": 0.00, "correction_squeezing": 0.504, "correction_displacement": 2.703j, "paper_p_trial": 3.80e-4, "paper_fidelity": 0.995},
    {"label": "Table I CPS (3,3)", "family": "cps", "m": 3, "n": 3, "g": 2.34, "alpha_in": -3.933j, "target_cubicity": 0.128, "target_squeezing": 0.00, "correction_squeezing": 0.527, "correction_displacement": 3.020j, "paper_p_trial": 6.69e-7, "paper_fidelity": 0.992},
    {"label": "Table I CPS (4,4)", "family": "cps", "m": 4, "n": 4, "g": 2.47, "alpha_in": -3.960j, "target_cubicity": 0.166, "target_squeezing": -0.01, "correction_squeezing": 0.647, "correction_displacement": 3.258j, "paper_p_trial": 1.19e-7, "paper_fidelity": 0.987},
]

print(f"tanh(rho_prime) = {math.sqrt(1.0 - 1.0 / G_PREP**2):.8f}")
print(f"ETA_PREP={ETA_PREP:.2f}, ETA_OUT={ETA_OUT:.2f}, N_DIM={N_DIM}, J_MAX={J_MAX}, L_MAX={L_MAX}")
print(f"Configured cases: {len(CASES)}; TOP_CHANNELS_PER_CASE={TOP_CHANNELS_PER_CASE}")


tanh(rho_prime) = 0.30004387
ETA_PREP=0.90, ETA_OUT=0.90, N_DIM=50, J_MAX=8, L_MAX=14
Configured cases: 40; TOP_CHANNELS_PER_CASE=0


## 论文的态与条件概率

`opa_heralded_state(signal, j, l, gain)` 返回第二篇论文式 (3)-(4) 的未归一化信号态 $|\Psi_{\psi,j,l}\rangle$。其范数平方就是

$$
P_{\psi}(g,j,l)=\|\Psi_{\psi,j,l}\|^2.
$$

后续对目标态施加的高斯校正是幺正操作，不改变这一成功概率；它只影响保真度的参照目标。对 CPS，目标态遵循论文式 (13)，并使用 Table I 的 $S(r_{\rm corr})D(\beta_{\rm corr})$ 校正。

In [3]:
# 3. 第二篇论文式 (3)、(4)、(6)、(13)、(18)
def cat_state(dim: int, alpha: float, parity: str) -> qt.Qobj:
    if parity not in {"odd", "even"}:
        raise ValueError("parity must be 'odd' or 'even'.")
    sign = -1.0 if parity == "odd" else 1.0
    return (qt.coherent(dim, alpha) + sign * qt.coherent(dim, -alpha)).unit()


def attenuate_fock_amplitudes(state: qt.Qobj, gain: float) -> qt.Qobj:
    amplitudes = state.full().ravel()
    factors = gain ** (-np.arange(len(amplitudes)))
    return qt.Qobj((amplitudes * factors).reshape((-1, 1)), dims=state.dims)


def h_coefficient(k: int, m: int, n: int, gain: float) -> float:
    added = n - m + k
    if added < 0:
        return 0.0
    G = math.sqrt(1.0 - 1.0 / gain**2)
    numerator = ((-1) ** added) * G ** (n - m + 2 * k)
    numerator *= math.sqrt(math.factorial(n) * math.factorial(m))
    denominator = gain ** (m - k + 1)
    denominator *= math.factorial(k) * math.factorial(added) * math.factorial(m - k)
    return numerator / denominator


def opa_heralded_state(signal: qt.Qobj, m_actual: int, n_actual: int, gain: float) -> qt.Qobj:
    """Second-paper Eq. (3): unnormalized conditional signal state."""
    if gain < 1.0:
        raise ValueError("OPA gain must satisfy g >= 1.")
    a = qt.destroy(signal.shape[0])
    output = 0.0 * signal
    for k in range(m_actual + 1):
        added = n_actual - m_actual + k
        coefficient = h_coefficient(k, m_actual, n_actual, gain)
        if coefficient != 0.0:
            output += coefficient * (a.dag() ** added) * attenuate_fock_amplitudes((a ** k) * signal, gain)
    return output


def squeezed_cat_target(dim: int, alpha: float, gamma: float, parity: str) -> qt.Qobj:
    return (qt.squeeze(dim, gamma) * cat_state(dim, alpha, parity)).unit()


def cubic_phase_target(dim: int, cubicity: float, squeezing: float) -> qt.Qobj:
    """Table-I CPS target from Eq. (13), using the manuscript's existing numerical convention x=a+a^dagger."""
    a = qt.destroy(dim)
    x = a + a.dag()
    cubic_phase = (1j * cubicity * (x ** 3)).expm()
    return (cubic_phase * qt.squeeze(dim, squeezing) * qt.basis(dim, 0)).unit()


def gaussian_correction_operator(dim: int, squeezing: float, displacement: complex) -> qt.Qobj:
    """Table-I correction S(r_corr) D(beta_corr), matching the original CPS calculation code."""
    return qt.squeeze(dim, squeezing) * qt.displace(dim, displacement)


## 两级有限效率 PNRD 模型

若真实有 $q$ 个光子而 detector 登记为 $r$ 个，二项 PNRD 权重是

$$
w_\eta(q\to r)=\binom{q}{r}\eta^r(1-\eta)^{q-r}.
$$

完整函数对真实 $(j,l)$ 通道求和，并返回未归一化条件密度矩阵。取其迹得到原始注册概率；归一化后才能计算有限效率保真度。

In [4]:
# 4. 成功概率与有限效率混态
def fock_preparation_probability(j_actual: int, g_prep: float) -> float:
    """Auxiliary SPDC probability P_Fock(j,g_prime)."""
    return (1.0 / g_prep**2) * (1.0 - 1.0 / g_prep**2) ** j_actual


def pnrd_weight(true_count: int, registered_count: int, efficiency: float) -> float:
    """Probability that true_count photons are registered as registered_count."""
    if true_count < registered_count:
        return 0.0
    return math.comb(true_count, registered_count) * efficiency**registered_count * (1.0 - efficiency) ** (true_count - registered_count)


def real_scalar(value) -> float:
    return float(np.real_if_close(value))


def evaluate_case(case: dict, eta_prep: float, eta_out: float, j_max: int, l_max: int) -> tuple[dict, list[dict]]:
    """Return simple and two-stage probabilities, conditional fidelity, and channel contributions."""
    m, n, gain = case["m"], case["n"], case["g"]
    family = case.get("family", "cat")
    if family == "cps":
        signal = qt.coherent(N_DIM, case["alpha_in"])
        target = cubic_phase_target(N_DIM, case["target_cubicity"], case["target_squeezing"])
        correction = gaussian_correction_operator(
            N_DIM, case["correction_squeezing"], case["correction_displacement"]
        )
    elif family == "cat":
        signal = cat_state(N_DIM, case["alpha_in"], case["input_parity"])
        target = squeezed_cat_target(N_DIM, case["target_alpha"], case["target_gamma"], case["target_parity"])
        correction = qt.qeye(N_DIM)
    else:
        raise ValueError(f"Unknown state family: {family}")
    state_cache: dict[tuple[int, int], qt.Qobj] = {}

    def conditional_state(j_actual: int, l_actual: int) -> qt.Qobj:
        key = (j_actual, l_actual)
        if key not in state_cache:
            state_cache[key] = opa_heralded_state(signal, j_actual, l_actual, gain)
        return state_cache[key]

    # This is exactly the original second-paper probability convention.
    ideal_state = conditional_state(m, n)
    p_gen_ideal = real_scalar(ideal_state.overlap(ideal_state))
    p_fock_ideal = fock_preparation_probability(m, G_PREP)
    p_trial_simple = eta_out * p_fock_ideal * p_gen_ideal
    corrected_ideal_state = correction * ideal_state
    fidelity_ideal_table_parameters = abs(target.overlap(corrected_ideal_state.unit())) ** 2

    rho_unnormalized = 0.0 * qt.ket2dm(qt.basis(N_DIM, 0))
    channel_rows: list[dict] = []
    for j_actual in range(m, j_max + 1):
        prep_factor = pnrd_weight(j_actual, m, eta_prep) * fock_preparation_probability(j_actual, G_PREP)
        if prep_factor == 0.0:
            continue
        for l_actual in range(n, l_max + 1):
            output_factor = pnrd_weight(l_actual, n, eta_out)
            if output_factor == 0.0:
                continue
            ket = conditional_state(j_actual, l_actual)
            p_ideal_jl = real_scalar(ket.overlap(ket))
            contribution = prep_factor * output_factor * p_ideal_jl
            if contribution == 0.0:
                continue
            corrected_ket = correction * ket
            rho_unnormalized += prep_factor * output_factor * qt.ket2dm(corrected_ket)
            channel_rows.append({
                "label": case["label"], "m_registered": m, "n_registered": n,
                "j_actual": j_actual, "l_actual": l_actual,
                "prep_weight": pnrd_weight(j_actual, m, eta_prep),
                "P_fock_j": fock_preparation_probability(j_actual, G_PREP),
                "out_weight": output_factor, "P_psi_j_l": p_ideal_jl,
                "contribution": contribution,
                "is_ideal_channel": (j_actual == m and l_actual == n),
            })

    p_trial_two_stage = real_scalar(rho_unnormalized.tr())
    if p_trial_two_stage <= 0.0:
        raise RuntimeError("Registered-event probability is zero; increase cutoffs or check parameters.")
    rho_conditioned = rho_unnormalized / p_trial_two_stage
    fidelity_eta = real_scalar(target.dag() * rho_conditioned * target)

    for row in channel_rows:
        row["fraction_of_registered_events"] = row["contribution"] / p_trial_two_stage
    channel_rows.sort(key=lambda row: row["contribution"], reverse=True)

    summary = {
        "label": case["label"], "family": family, "m": m, "n": n, "g": gain,
        "alpha_in": case["alpha_in"], "input_parity": case.get("input_parity", "coherent"),
        "eta_prep": eta_prep, "eta_out": eta_out,
        "P_fock_m": p_fock_ideal, "P_gen_ideal": p_gen_ideal,
        "P_trial_simple": p_trial_simple,
        "P_trial_two_stage": p_trial_two_stage,
        "F_ideal_table_parameters": float(fidelity_ideal_table_parameters),
        "F_conditional_eta": fidelity_eta,
        "paper_P_trial": case["paper_p_trial"],
        "P_simple_to_paper_ratio": p_trial_simple / case["paper_p_trial"],
        "paper_F_ideal": case["paper_fidelity"],
        "ideal_channel_fraction": next((row["fraction_of_registered_events"] for row in channel_rows if row["is_ideal_channel"]), 0.0),
    }
    return summary, channel_rows


In [5]:
# 5. 运行计算、基础一致性检查和结果打印
summary_rows, contribution_rows = [], []

for case in CASES:
    summary, contributions = evaluate_case(case, ETA_PREP, ETA_OUT, J_MAX, L_MAX)
    summary_rows.append(summary)
    contribution_rows.extend(contributions)

    # Independent limiting-case check: both perfect detectors must recover P_Fock(m)*P_psi(m,n).
    perfect, _ = evaluate_case(case, 1.0, 1.0, case["m"], case["n"])
    expected = summary["P_fock_m"] * summary["P_gen_ideal"]
    if not np.isclose(perfect["P_trial_two_stage"], expected, rtol=1e-12, atol=1e-14):
        raise AssertionError(f"Perfect-detector limit failed for {case['label']}.")

print("=" * 152)
print("Finite-efficiency PNRD analysis")
print("=" * 152)
print(f"{'case':<36} {'P_simple':>13} {'P_two-stage':>13} {'F_ideal':>10} {'F_eta':>10} {'ideal channel':>15} {'simple/paper':>13}")
print("-" * 152)
for row in summary_rows:
    print(
        f"{row['label']:<36} {row['P_trial_simple']:>13.5e} {row['P_trial_two_stage']:>13.5e} "
        f"{row['F_ideal_table_parameters']:>10.6f} {row['F_conditional_eta']:>10.6f} "
        f"{row['ideal_channel_fraction']:>14.2%} {row['P_simple_to_paper_ratio']:>13.5f}"
    )

probability_mismatches = [row for row in summary_rows if not np.isclose(row["P_simple_to_paper_ratio"], 1.0, rtol=0.02)]
if probability_mismatches:
    print("\nWarning: the following simple probabilities differ from the printed paper table by more than 2%:")
    for row in probability_mismatches:
        print(
            f"  {row['label']}: P_simple={row['P_trial_simple']:.5e}, "
            f"paper={row['paper_P_trial']:.5e}, ratio={row['P_simple_to_paper_ratio']:.5f}"
        )

if TOP_CHANNELS_PER_CASE > 0:
    print("\nMost important true (j,l) channels")
    for case in CASES:
        print(f"\n{case['label']}")
        top_rows = [row for row in contribution_rows if row["label"] == case["label"]][:TOP_CHANNELS_PER_CASE]
        for row in top_rows:
            print(
                f"  true (j,l)=({row['j_actual']},{row['l_actual']}), "
                f"share={row['fraction_of_registered_events']:.2%}, "
                f"contribution={row['contribution']:.5e}"
            )


Finite-efficiency PNRD analysis
case                                      P_simple   P_two-stage    F_ideal      F_eta   ideal channel  simple/paper
--------------------------------------------------------------------------------------------------------------------------------------------------------
Table II OCS (1,1), alpha=0.30         1.38854e-03   1.70887e-03   0.999962   0.746321         73.13%       0.99895
Table II OCS (1,1), alpha=1.06         2.73865e-03   3.05393e-03   0.992161   0.814965         80.71%       1.00317
Table II OCS (1,1), alpha=1.31         3.94370e-03   4.27274e-03   0.976136   0.817475         83.07%       1.00094
Table II OCS (2,2), alpha=0.30         2.77730e-04   2.67425e-04   0.999972   0.773509         75.71%       0.99903
Table II OCS (2,2), alpha=1.06         3.33020e-04   3.23277e-04   0.999331   0.773193         75.10%       1.00006
Table II OCS (2,2), alpha=1.31         3.71555e-04   3.65043e-04   0.995044   0.758870         74.20%       1.00149
Ta

In [6]:
# 6. 保存完整结果。修改 ETA 或截断后会覆盖这些 CSV。
summary_columns = list(summary_rows[0].keys())
with (OUTPUT_DIR / "summary.csv").open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=summary_columns)
    writer.writeheader()
    writer.writerows(summary_rows)

contribution_columns = list(contribution_rows[0].keys())
with (OUTPUT_DIR / "channel_contributions.csv").open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=contribution_columns)
    writer.writeheader()
    writer.writerows(contribution_rows)

print(f"\nSaved: {(OUTPUT_DIR / 'summary.csv').resolve()}")
print(f"Saved: {(OUTPUT_DIR / 'channel_contributions.csv').resolve()}")



Saved: C:\Users\10260\second code combination\outputs\second_paper_finite_efficiency_probability_outputs\summary.csv
Saved: C:\Users\10260\second code combination\outputs\second_paper_finite_efficiency_probability_outputs\channel_contributions.csv


In [7]:
# 7. 截断收敛检查：比较当前截断与更大的截断。
# 对发表或正式回复，请确认相对差异足够小；若不小，应继续提高 J_MAX、L_MAX 和/或 N_DIM。
CHECK_J_MAX = J_MAX + 2
CHECK_L_MAX = L_MAX + 4

print(f"Comparing (J_MAX,L_MAX)=({J_MAX},{L_MAX}) with ({CHECK_J_MAX},{CHECK_L_MAX})")
for case, base in zip(CASES, summary_rows):
    enlarged, _ = evaluate_case(case, ETA_PREP, ETA_OUT, CHECK_J_MAX, CHECK_L_MAX)
    rate_relative_change = abs(enlarged["P_trial_two_stage"] - base["P_trial_two_stage"]) / base["P_trial_two_stage"]
    fidelity_absolute_change = abs(enlarged["F_conditional_eta"] - base["F_conditional_eta"])
    print(
        f"{case['label']}: relative change in P_two-stage={rate_relative_change:.3e}, "
        f"absolute change in F_eta={fidelity_absolute_change:.3e}"
    )


Comparing (J_MAX,L_MAX)=(8,14) with (10,18)
Table II OCS (1,1), alpha=0.30: relative change in P_two-stage=1.269e-16, absolute change in F_eta=0.000e+00
Table II OCS (1,1), alpha=1.06: relative change in P_two-stage=3.238e-14, absolute change in F_eta=2.631e-14
Table II OCS (1,1), alpha=1.31: relative change in P_two-stage=4.994e-14, absolute change in F_eta=4.075e-14
Table II OCS (2,2), alpha=0.30: relative change in P_two-stage=3.852e-15, absolute change in F_eta=2.998e-15
Table II OCS (2,2), alpha=1.06: relative change in P_two-stage=6.419e-13, absolute change in F_eta=4.963e-13
Table II OCS (2,2), alpha=1.31: relative change in P_two-stage=1.707e-12, absolute change in F_eta=1.295e-12
Table II OCS (3,3), alpha=0.30: relative change in P_two-stage=3.254e-11, absolute change in F_eta=1.556e-11
Table II OCS (3,3), alpha=1.06: relative change in P_two-stage=9.733e-10, absolute change in F_eta=6.527e-10
Table II OCS (3,3), alpha=1.31: relative change in P_two-stage=5.714e-10, absolute c

## 如何使用结果

- `P_trial_simple`：原第二篇表格的简化口径。它只使用主输出端的整体效率 `ETA_OUT`，应与论文 Tables I-IV 的 $P_{\rm trial}$ 对照。
- `P_trial_two_stage`：两个有限效率 PNRD 都考虑后的登记概率；它包含错误 heralding 通道。
- `F_ideal_table_parameters`：探测效率为 1 时，使用论文表中四舍五入参数重新计算的保真度。CPS 使用 Table I 的高斯校正与式 (13) 目标态；猫态使用 Tables II-IV 的压缩猫目标态。
- `F_conditional_eta`：在登记到目标 $(m,n)$ 的所有事件中，条件信号混态对固定目标态的保真度。
- `ideal_channel_fraction`：所有登记事件中真正来自 $(j,l)=(m,n)$ 的比例。该数越小，错误通道的混合通常越显著。

### Table I 的一个待核对数值

在保持 Table I 所列参数不变时，CPS $(m,n)=(1,5)$ 的简化概率计算为 $3.74955\times10^{-3}$，而表中印为 $3.75\times10^{-4}$，相差十倍。该 notebook 保留表中印刷数值于 `paper_P_trial`，并额外输出 `P_simple_to_paper_ratio`，不会静默修改论文数据。此处应在修订稿前回查原始计算记录。

若要把探测效率从 $0.90$ 改成 $0.95$，仅修改第 2 个代码单元：

```python
ETA_PREP = 0.95
ETA_OUT = 0.95
```

然后依次运行所有代码单元。保留 `P_trial_simple` 与 `P_trial_two_stage` 两列，不能直接把旧的 `P_trial_simple` 再额外乘以 $0.95$；这会重复计入输出端效率。

若只希望模拟主 OPA 输出端 detector 的效率变化，而认为输入 $|m\rangle$ 是确定性高保真来源，则应另建“固定真实输入 $m$”模型：不对 $j$ 求和，并把前端制备概率从结果中移除或按实际来源重新定义。